# 01 · LandingBaixa os parquets originais da NYC TLC para o Volume da landing zone.Nada é transformado aqui: o arquivo é gravado byte a byte como veio da fonte.É o que garante que qualquer erro nas camadas seguintes possa ser corrigidosem depender do site da TLC continuar no ar.Layout gravado:```/Volumes/ifood_case/landing/files/ny_taxi_trip/  yellow/ref_year=2023/ref_month=01/data.parquet  yellow/ref_year=2023/ref_month=02/data.parquet  ...  green/ref_year=2023/ref_month=05/data.parquet```

In [ ]:
%load_ext autoreload%autoreload 2

In [ ]:
# Coloca a raiz do repositório no sys.path.# Sobe diretórios até achar a pasta `src`, de modo que o notebook funcione# tanto em Git Folders quanto em Workspace Files, sem caminho hardcoded.import osimport sys_root = os.getcwd()while _root != "/" and not os.path.isdir(os.path.join(_root, "src")):    _root = os.path.dirname(_root)if _root not in sys.path:    sys.path.insert(0, _root)print("Repo root:", _root)

In [ ]:
from src import configfrom src.ingestion.ny_taxi_trip_extractor import NyTaxiTripExtractorextractor = NyTaxiTripExtractor()resultados = []for trip_type in config.TRIP_TYPES:    resultados += extractor.extract(        trip_type,        start=config.INGESTION_START,        end=config.INGESTION_END,        overwrite=True,    )

## Resumo da extração

In [ ]:
import pandas as pdresumo = pd.DataFrame([r.__dict__ for r in resultados])resumo['size_mb'] = (resumo['size_bytes'] / 1024 / 1024).round(1)display(resumo[['trip_type', 'year', 'month', 'status', 'size_mb', 'destination']])

In [ ]:
# Falha explícita se algum mês da janela não chegou na landing.faltando = [r for r in resultados if r.status == 'not_found']assert not faltando, f'Meses ausentes na origem: {faltando}'print(f'{len(resultados)} arquivos disponíveis na landing.')

---## Plano B — upload manualO Databricks Free Edition **restringe o acesso de saída à internet** a umconjunto de domínios confiáveis. O CDN da TLC não está nessa lista, então acélula de download acima falha a menos que a conta tenha passado pelaverificação com LinkedIn (que libera acesso externo).Se o download automático falhar, siga este caminho:1. Rode a célula abaixo para imprimir os 10 links e baixe os arquivos no seu   computador (é só abrir cada link no navegador).2. No menu lateral: **Catalog → ifood_case → landing → files → Upload**, e envie   os 10 arquivos para uma pasta chamada `uploads`.3. Rode a última célula, que organiza os arquivos no layout particionado que o   restante do pipeline espera.O resultado é idêntico ao do download automático — as camadas seguintes nãosabem por qual caminho o arquivo chegou.

In [ ]:
from src import configfrom src.ingestion.ny_taxi_trip_extractor import NyTaxiTripExtractor, month_rangeextractor = NyTaxiTripExtractor()print('Baixe estes arquivos e envie para a pasta `uploads` do Volume:\n')for trip_type in config.TRIP_TYPES:    for ano, mes in month_range(config.INGESTION_START, config.INGESTION_END):        print(extractor.build_url(trip_type, ano, mes))

In [ ]:
# Organiza os arquivos enviados manualmente no layout particionado da landing.import osimport reimport shutilfrom src import configUPLOAD_DIR = f"/Volumes/{config.CATALOG}/{config.SCHEMA_LANDING}/{config.LANDING_VOLUME}/uploads"PADRAO = re.compile(r"(yellow|green)_tripdata_(\d{4})-(\d{2})\.parquet$")organizados = 0for nome in sorted(os.listdir(UPLOAD_DIR)):    match = PADRAO.search(nome)    if not match:        print(f"ignorado (fora do padrão): {nome}")        continue    trip_type, ano, mes = match.groups()    destino_dir = config.landing_path(trip_type, ano, mes)    os.makedirs(destino_dir, exist_ok=True)    shutil.copyfile(f"{UPLOAD_DIR}/{nome}", f"{destino_dir}/data.parquet")    print(f"ok: {nome} -> {destino_dir}/data.parquet")    organizados += 1assert organizados == 10, f"Esperados 10 arquivos, organizados {organizados}. Confira a pasta uploads."print(f"\n{organizados} arquivos organizados na landing.")